# Linear regression

_Machine Learning_

**Fit a straight line through your data. The simplest predictor.**

Linear regression predicts a number with a straight line (or flat plane in many dimensions).

     The prediction is a dot product of weights and features.

---

This notebook is a practice scaffold for the **Linear regression** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. Each row is one example; the columns are its features, and the target is a continuous value.

In [ ]:
from sklearn.datasets import load_diabetes

data = load_diabetes(as_frame=True)
print("rows x columns:", data.frame.shape)
print("feature columns:", list(data.data.columns))
print("target summary:")
print(data.target.describe())
data.frame.head()

## Reference implementation — NumPy + scikit-learn

### Step 1 — Make synthetic data from a known line

We generate 50 points from `y = 3x + 2` and add a little Gaussian noise. Because we know the true slope (3) and intercept (2), we can later check whether our fit recovers them. The random generator is seeded so the data is reproducible.

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression

# synthetic data: y = 3x + 2 + noise
rng = np.random.default_rng(0)
X = rng.uniform(0, 10, size=(50, 1))
y = 3 * X[:, 0] + 2 + rng.normal(0, 1, size=50)

### Step 2 — Solve with the normal equation

The least-squares weights have a closed form: `w = (Xᵀ X)⁻¹ Xᵀ y`. We first prepend a column of 1s to `X` so the model can learn an intercept (the bias term). Then we solve the linear system directly with `np.linalg.solve`, which is more stable than inverting the matrix by hand. The two entries of `w` are the intercept and the slope.

In [ ]:
# closed-form: w = (X^T X)^-1 X^T y
Xb = np.c_[np.ones(len(X)), X]
w = np.linalg.solve(Xb.T @ Xb, Xb.T @ y)
print("normal equation  intercept=%.3f  slope=%.3f" % (w[0], w[1]))

### Step 3 — Cross-check with scikit-learn

scikit-learn's `LinearRegression` solves the same least-squares problem under the hood. Its intercept and slope should match the normal-equation values, and both should land near the true 2 and 3. The `R²` score reports how much of the variance in `y` the line explains (1.0 is a perfect fit).

In [ ]:
# scikit-learn
m = LinearRegression().fit(X, y)
print("scikit-learn     intercept=%.3f  slope=%.3f" % (m.intercept_, m.coef_[0]))
print("R^2 =", round(m.score(X, y), 4))

## Visualize the data & results

_Does a straight line fit diabetes progression from body-mass index, and what is the slope?_

### Step 1 — Fit a line to real diabetes data

We switch to 442 real patients and predict disease progression from a single feature: body-mass index (already standardized in this dataset). We slice out just the BMI column and fit a `LinearRegression`. The learned slope tells us how strongly progression rises with BMI.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes
from sklearn.linear_model import LinearRegression

# 442 real diabetes patients; predict progression from BMI
db = load_diabetes()
X = db.data[:, 2:3]                # BMI feature (standardized)
y = db.target
m = LinearRegression().fit(X, y)
print("slope", m.coef_[0], "intercept", m.intercept_)

### Step 2 — Compute the fitted line's endpoints

To draw the regression line we only need two points: its value at the smallest BMI and at the largest BMI. We build those two x-values and ask the model to predict their y-values, giving us the endpoints of the straight line to plot.

In [ ]:
xs = np.array([[X.min()], [X.max()]])
ys = m.predict(xs)

### Step 3 — Plot the patients and the fitted line

We scatter every patient (BMI vs progression) and overlay the fitted line. The upward-sloping line shows that higher BMI is associated with greater disease progression, though the scatter around the line reminds us BMI alone does not fully determine the outcome.

In [ ]:
plt.scatter(X[:, 0], y, c="#4ea1ff", edgecolor="k", label="patients")
plt.plot(xs[:, 0], ys, color="#ffb454", label="fit")
plt.xlabel("BMI (standardized)")
plt.ylabel("disease progression")
plt.title("Least-squares fit: BMI to progression")
plt.legend()
plt.show()